# 03 · Calibrated timing (14 & 30 days) and a price model

The last modelling step before the demo. Three questions:

1. **Two horizons.** P(sale within **14** days) and within **30** days. Same LightGBM setup as notebook 02.
2. **Calibration.** When the model says *70%*, does a sale happen about 70% of the time? The website shows these numbers as **confidence**, so they have to be honest. We check with a reliability diagram and fix it with **isotonic calibration**.
3. **How deep?** Given that a sale happens, what discount? A **quantile regression** gives a *range* ("probably 50–75% off") rather than a single guess, scored with the regression metrics from **Lecture 10**.

**Three-way time split** (Lecture 19, plus a calibration slice):

| Part | Check-in dates | Used for |
|---|---|---|
| fit | before 2024 | training the models |
| calibrate | 2024 | fitting the calibration mapping |
| test | 2025 onward | final scores, never touched before |

Calibrating on the same data the model was trained on would make it look better calibrated than it really is, which is why 2024 gets its own slice.

Features come from `steamwait/features.py`, the **same code `predict.py` will use** for the website.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, mean_absolute_error, roc_auc_score

from steamwait import features as F

pd.set_option("display.max_columns", 30)
con = duckdb.connect(str(ROOT / "data" / "warehouse.duckdb"), read_only=True)

FIT_END = pd.Timestamp("2024-01-01")
CAL_END = pd.Timestamp("2025-01-01")
event_dates = F.load_event_dates(con)
gaps = F.load_gaps(con)
print(f"{len(gaps):,} gaps, {gaps['appid'].nunique():,} games, {len(event_dates)} major events")

## Part A: timing at 14 and 30 days

For each horizon: build check-in rows, split three ways, and drop any row whose label window crosses into the next slice (it would peek ahead).

In [ ]:
def split(df, horizon):
    end = df["check_date"] + pd.to_timedelta(horizon, unit="D")
    fit = df[end < FIT_END]
    cal = df[(df["check_date"] >= FIT_END) & (end < CAL_END)]
    test = df[df["check_date"] >= CAL_END]
    return [F.add_game_weights(p.copy()) for p in (fit, cal, test)]

def X(df):
    return df[F.TIMING_NUMERIC + F.TIMING_CATEGORICAL].astype({"check_month": "category"})

def make_lgbm():
    return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              random_state=330, verbose=-1)

timing, scores = {}, {}
for h in (14, 30):
    rows = F.build_checkins(gaps, event_dates, horizon=h)
    fit, cal, test = split(rows, h)
    model = make_lgbm().fit(X(fit), fit["label"], sample_weight=fit["weight"])
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(
        model.predict_proba(X(cal))[:, 1], cal["label"], sample_weight=cal["weight"])
    p_raw = model.predict_proba(X(test))[:, 1]
    p_cal = iso.predict(p_raw)
    rule = (test["days_to_next_major_event"] <= h).astype(float)
    w, y = test["weight"], test["label"]
    timing[h] = dict(model=model, iso=iso, test=test, p_raw=p_raw, p_cal=p_cal)
    for name, p in [("Seasonal rule", rule), ("LightGBM", p_raw), ("LightGBM + calibration", p_cal)]:
        scores[(f"{h} days", name)] = {
            "test AP": average_precision_score(y, p, sample_weight=w),
            "test ROC AUC": roc_auc_score(y, p, sample_weight=w),
            "Brier (lower = better)": brier_score_loss(y, p, sample_weight=w),
            "positive rate": np.average(y, weights=w),
        }
    print(f"{h}d: fit {len(fit):,} | calibrate {len(cal):,} | test {len(test):,} rows")

pd.DataFrame(scores).T.round(3)

**How to read this:** calibration is a monotone re-mapping, so it shouldn't change the *ranking* (AP and ROC AUC stay about the same). What it should improve is the **Brier score**, which punishes confident wrong answers. The seasonal rule only outputs 0 or 1, so its Brier score is poor by design.

## Calibration check: does "70%" mean 70%?

A **reliability diagram** groups test predictions into bins (0–10%, 10–20%, …) and compares the average prediction with how often a sale actually happened. Perfect calibration falls on the diagonal.

In [ ]:
def reliability(p, y, w, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    rows = []
    for b in range(bins):
        m = idx == b
        if w[m].sum() > 0:
            rows.append((np.average(p[m], weights=w[m]), np.average(y[m], weights=w[m]), m.sum()))
    return pd.DataFrame(rows, columns=["predicted", "actual", "n"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, h in zip(axes, (14, 30)):
    t = timing[h]
    y, w = t["test"]["label"].to_numpy(), t["test"]["weight"].to_numpy()
    for label, p in [("raw LightGBM", t["p_raw"]), ("calibrated", t["p_cal"])]:
        r = reliability(p, y, w)
        ax.plot(r["predicted"], r["actual"], marker="o", label=label)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="perfect")
    ax.set(title=f"Sale within {h} days (test, 2025+)", xlabel="predicted probability",
           ylabel="actual share that went on sale", xlim=(0, 1), ylim=(0, 1))
    ax.legend()
plt.tight_layout()

## Part B: how deep will the sale be?

One row per sale, with features known **when it starts**: the game's previous discounts (last, median of the last three, deepest so far), whether it starts with a big platform sale, price, age and month.

- **Baseline (Lecture 2 idea):** *"same discount as last time."*
- **Model:** LightGBM **quantile regression** at the 10th, 50th and 90th percentiles. The 50th is the point guess, and 10th–90th is the range shown on the website.
- **Metrics (Lecture 10):** MAE in percentage points, plus how often the true discount falls inside the 10–90 range (ideally about 80%).

In [ ]:
depth = F.build_depth_rows(con, event_dates)
depth = depth[depth["n_prev_sales"] >= 1]          # need at least one past sale to learn from
d_fit = F.add_game_weights(depth[depth["start_date"] < CAL_END].copy())
d_test = F.add_game_weights(depth[depth["start_date"] >= CAL_END].copy())
Xd = lambda df: df[F.DEPTH_FEATURES].astype(float)
print(f"depth rows: fit {len(d_fit):,} | test {len(d_test):,}")

quantiles = {}
for a in (0.1, 0.5, 0.9):
    quantiles[a] = lgb.LGBMRegressor(objective="quantile", alpha=a, n_estimators=400, learning_rate=0.05,
                                     num_leaves=31, min_child_samples=50, random_state=330, verbose=-1) \
        .fit(Xd(d_fit), d_fit["max_cut_percent"], sample_weight=d_fit["weight"])

y, w = d_test["max_cut_percent"], d_test["weight"]
q10, q50, q90 = (np.clip(quantiles[a].predict(Xd(d_test)), 0, 100) for a in (0.1, 0.5, 0.9))
baseline = d_test["last_cut"]
inside = ((y >= q10) & (y <= q90)).astype(float)

pd.DataFrame({
    "MAE (percentage points)": [mean_absolute_error(y, baseline, sample_weight=w),
                                mean_absolute_error(y, q50, sample_weight=w)],
    "share inside 10-90% range": [np.nan, np.average(inside, weights=w)],
    "avg range width (points)": [np.nan, np.average(q90 - q10, weights=w)],
}, index=["Baseline: same as last sale", "LightGBM quantile (median)"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
s = d_test.sample(min(3000, len(d_test)), random_state=1).index
pos = d_test.index.get_indexer(s)
ax.scatter(y.loc[s], q50[pos], alpha=0.15, s=8)
ax.plot([0, 100], [0, 100], "k--", alpha=0.5)
ax.set(title="Predicted vs actual discount (test sales)", xlabel="actual discount %",
       ylabel="predicted (median) %", xlim=(0, 100), ylim=(0, 100));

## Putting it together: what the website will say

For a few test check-ins, combine **when** (calibrated 14/30-day probabilities, plus the next big sale) with **how much** (discount range turned into a price range). `predict.py` will produce exactly this for every game, as of today.

In [ ]:
t30 = timing[30]["test"].assign(p30=timing[30]["p_cal"])
ex = t30.sample(6, random_state=42).copy()
ex["p14"] = timing[14]["iso"].predict(timing[14]["model"].predict_proba(X(ex))[:, 1])

# depth features for "if it went on sale soon": reuse the game's sale history so far
last = depth.sort_values("start_date").groupby("appid").tail(1).set_index("appid")
ex = ex.join(last[["last_cut", "median_last3_cut", "max_cut_so_far", "n_prev_sales"]], on="appid")
ex["days_since_prev_sale"] = ex["days_waited"]
ex["is_major_event_sale"] = (ex["days_to_next_major_event"] <= 30).astype(int)
ex["start_month"] = ex["check_month"]
dX = ex[F.DEPTH_FEATURES].astype(float)
lo, mid, hi = (np.clip(quantiles[a].predict(dX), 0, 100) for a in (0.1, 0.5, 0.9))

pd.DataFrame({
    "game": ex["name"],
    "as of": ex["check_date"].dt.date,
    "P(sale ≤14d)": ex["p14"].round(2),
    "P(sale ≤30d)": ex["p30"].round(2),
    "next big sale": ex["next_major_event"].dt.date,
    "regular $": ex["regular_dollars"],
    "likely discount": [f"{a:.0f}-{b:.0f}%" for a, b in zip(lo, hi)],
    "projected price": [f"${r*(1-b/100):.2f}-${r*(1-a/100):.2f}" for r, a, b in zip(ex["regular_dollars"], lo, hi)],
    "actually (30d)": ex["label"].map({1: "sale", 0: "no sale"}),
}).reset_index(drop=True)

## Takeaways (fill in from your run)

- **30 vs 14 days:** _how much easier is the 30-day question? (Compare AP against each one's positive rate.)_
- **Calibration:** _was raw LightGBM over- or under-confident? Did isotonic calibration pull the curve onto the diagonal and lower the Brier score?_
- **Discount depth:** _did the quantile model beat "same as last sale" on MAE? Does about 80% of real discounts fall inside the 10–90 range?_

**Next:** `predict.py` trains these same models on **all** history and writes `predictions.json` for every game as of today, ready for the website.